In [62]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'meta-llama/Llama-3.2-1B-Instruct'
max_length = 512
debug = 4

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

In [55]:
# Load dataset and prepare for training
from pathlib import Path
import jsonlines
from datasets import load_from_disk

train_dataset = load_from_disk("datasets/splits/train")

In [ ]:
import torch


def create_labels_for_all(input_ids, attention_mask):
        """
        Create labels for all tokens except padding (mask those with -100).
        """
        labels = []
        for i, mask in enumerate(attention_mask):
            if mask == 0:  # Padding token
                labels.append(-100)
            else:
                labels.append(input_ids[i])
        return labels

def create_masked_labels(messages, tokenizer, input_ids, attention_mask):
    """Create labels with input tokens masked (-100)"""
    labels = [-100] * len(input_ids)
    
    # Mask padding tokens in labels
    for i, mask in enumerate(attention_mask):
        if mask == 0:  # Padding token
            labels[i] = -100
    
    # Find assistant responses and unmask only those tokens
    for msg in messages:
        if msg['role'] == 'assistant':
            assistant_content = msg['content']
            
            # Find where this assistant response appears in the tokenized text
            assistant_tokens = tokenizer.encode(assistant_content, add_special_tokens=False)
            
            # Find the position of assistant response in input_ids
            decoded_assistant = [tokenizer.decode(item) for item in assistant_tokens]
            decoded_input = [tokenizer.decode(item) for item in input_ids]
            for i in range(len(input_ids) - len(assistant_tokens) + 1):
                # Only check non-padding tokens
                if debug == 4 and torch.cuda.current_device() == 0:
                    print(f"=======input_ids: {input_ids[i:i+len(assistant_tokens)]}")
                    print(f"assistant_tokens: {assistant_tokens}")

                if attention_mask[i] == 1 and decoded_input[i:i+len(assistant_tokens)] == decoded_assistant:
                    # Unmask the assistant response tokens
                    for j in range(i, min(i + len(assistant_tokens), len(input_ids))):
                        if attention_mask[j] == 1:  # Only unmask non-padding tokens
                            labels[j] = input_ids[j]
                    break
            
            if debug == 4:
                exit(0)
    
    return labels

# Tokenize conversations
for msg_idx, msg in enumerate(train_dataset['messages']):
    # Format message
    formatted_msg = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
    # Tokenize
    tokenized_msg = tokenizer(
        formatted_msg,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors=None
    )

    input_ids = tokenized_msg['input_ids']
    attention_mask = tokenized_msg['attention_mask']
    print(tokenized_msg)
    break

{'input_ids': [128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 777, 3799, 220, 2366, 20, 271, 12281, 5933, 4221, 311, 264, 5410, 36018, 2082, 11, 902, 649, 387, 18407, 1937, 1507, 11, 36476, 26042, 477, 5125, 39539, 13, 128009, 128006, 882, 128007, 271, 2002, 354, 316, 7583, 264, 12088, 20782, 62150, 14200, 409, 92824, 2445, 8938, 1327, 3890, 1624, 4447, 320, 782, 2041, 16721, 8, 128009, 128006, 78191, 128007, 271, 19503, 1937, 1507, 220, 17079, 17252, 2371, 128009], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [43]:
train_dataset[0]

{'concept_id': 4140970,
 'prompt': 'tenotomía a cielo abierto de músculo extensor del pie (procedimiento)',
 'completion': 'SNOMED 33253004',
 '__index_level_0__': 844204,
 'messages': [{'content': 'Convert natural language to a standard vocabulary code, which can be SNOMED, RxNorm or LOINC.',
   'role': 'system'},
  {'content': 'tenotomía a cielo abierto de músculo extensor del pie (procedimiento)',
   'role': 'user'},
  {'content': 'SNOMED 33253004', 'role': 'assistant'}]}

In [39]:
dir(train_dataset)

['__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__ior__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__orig_bases__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed__',
 '__ror__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_check_values_features',
 '_check_values_type',
 'align_labels_with_mapping',
 'cache_files',
 'cast',
 'cast_column',
 'class_encode_column',
 'cleanup_cache_files',
 'clear',
 'column_names',
 'copy',
 'data',
 'filter',
 'flatten',
 'flatten_indices',
 'formatted_as',
 'from_csv',
 'from_json',
 'from_parquet',
 'from_text',
 'fromkeys',
 'get',
 'items',
 'keys',
 'load_from_disk',
 'map',
 'nu

In [28]:
i = 0
for data in train_dataset['train']:
    print(data)
    i += 1
    if i > 100:
        break


{'messages': [{'role': 'system', 'content': 'Convert natural language to a standard vocabulary code, which can be SNOMED, RxNorm or LOINC.'}, {'role': 'user', 'content': 'tenotomía a cielo abierto de músculo extensor del pie (procedimiento)'}, {'role': 'assistant', 'content': 'SNOMED 33253004'}]}
{'messages': [{'role': 'system', 'content': 'Convert natural language to a standard vocabulary code, which can be SNOMED, RxNorm or LOINC.'}, {'role': 'user', 'content': 'Ultrasound Doppler flow mapping of artery of lower limb'}, {'role': 'assistant', 'content': 'SNOMED 431217008'}]}
{'messages': [{'role': 'system', 'content': 'Convert natural language to a standard vocabulary code, which can be SNOMED, RxNorm or LOINC.'}, {'role': 'user', 'content': '3 days; Centers for Medicare and Medicaid Assessment; Disch; Pan; Panel; PANEL.SURVEY.CMS; Panl; Pnl; Survey'}, {'role': 'assistant', 'content': 'LOINC 86615-2'}]}
{'messages': [{'role': 'system', 'content': 'Convert natural language to a standar